In [31]:
import numpy as np
import torch
import yaml

from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded
from src.metrics import pehe, rmse

In [2]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

In [32]:
train_ds, val_ds, _, ytrain_std = load_ihdp(
    cfg.data.path,
    replication=1,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)

train_ds_conf, val_ds_conf = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect) for ds in (train_ds, val_ds)
)

In [35]:
def arbitrary_evaluation(train_ds, val_ds):
    all_x = torch.cat([train_ds.x, val_ds.x])
    all_a = torch.cat([train_ds.a, val_ds.a])
    all_y = torch.cat([train_ds.y, val_ds.y])

    all_mu0 = torch.cat([train_ds.mu0, val_ds.mu0])
    all_mu1 = torch.cat([train_ds.mu1, val_ds.mu1])

    a0_idx = all_a == 0
    a1_idx = all_a == 1
    print("Untreated subjects (a=0):", a0_idx.sum().item())
    print("Treated subjects (a=1):", a1_idx.sum().item())

    # Compute what metrics would look like under a completely trivial baseline:
    # y_cf = y_fac for every subject, no model at all

    fac_y0 = all_y[a0_idx]
    fac_y1 = all_y[a1_idx]
    cf_mu0 = all_mu0[a1_idx]
    cf_mu1 = all_mu1[a0_idx]

    # RMSE: per-arm, only needs the subjects where that arm is actually counterfactual
    rmse_y0, rmse_y1 = rmse(fac_y1.unsqueeze(1), fac_y0.unsqueeze(1), cf_mu0, cf_mu1)

    # PEHE: needs y0_hat/y1_hat/mu0/mu1 aligned to the SAME full subject set,
    # since CATE is a per-subject quantity (y1-y0 vs mu1-mu0)
    y_fac_all = all_y.unsqueeze(1)  # trivial: predict both PO slots = own factual y
    pehe_trivial = pehe(y_fac_all, y_fac_all, all_mu0, all_mu1)

    result_trivial = ytrain_std * np.array((rmse_y0, rmse_y1, pehe_trivial))

    # Population-mean baseline: predict the counterfactual arm's outcome as the
    # observed mean of that arm among subjects actually assigned to it -- no
    # individual-level information at all, and directly inherits confounding bias
    # (mean(y|a=1) != true E[Y1] when the confounder correlates with both).
    pop_mean_y0 = fac_y0.mean()
    pop_mean_y1 = fac_y1.mean()

    n_a1 = a1_idx.sum().item()
    n_a0 = a0_idx.sum().item()

    # RMSE: per-arm, only over the subjects where that arm is counterfactual
    rmse_y0, rmse_y1 = rmse(
        pop_mean_y0.expand(n_a1).unsqueeze(1),
        pop_mean_y1.expand(n_a0).unsqueeze(1),
        cf_mu0,
        cf_mu1,
    )

    # PEHE: needs a prediction for every subject, aligned to the full mu0/mu1 --
    # a genuinely naive predictor just outputs the same constant for everyone,
    # not only for whichever arm happens to be counterfactual for them
    B = len(all_x)
    y0_hat_all = pop_mean_y0.expand(B).unsqueeze(1)
    y1_hat_all = pop_mean_y1.expand(B).unsqueeze(1)
    pehe_trivial_popmean = pehe(y0_hat_all, y1_hat_all, all_mu0, all_mu1)

    result_popmean = ytrain_std * np.array((rmse_y0, rmse_y1, pehe_trivial_popmean))

    return result_trivial, result_popmean

In [36]:
result_trivial, result_popmean = arbitrary_evaluation(train_ds, val_ds)
print("Trivial baseline:", result_trivial)
print("Population-mean baseline:", result_popmean)

Untreated subjects (a=0): 517
Treated subjects (a=1): 320
Trivial baseline: [4.24369486 4.23192587 4.1169197 ]
Population-mean baseline: [1.22205154 0.46905742 0.86169098]


In [37]:
result_trivial_conf, result_popmean_conf = arbitrary_evaluation(train_ds_conf, val_ds_conf)
print("Trivial baseline (confounded):", result_trivial_conf)
print("Population-mean baseline (confounded):", result_popmean_conf)

Untreated subjects (a=0): 425
Treated subjects (a=1): 412
Trivial baseline (confounded): [3.9909227  4.00018389 3.8500903 ]
Population-mean baseline (confounded): [1.55793724 0.47114698 1.15694747]
